# 07 — Silver Bureau Balance

**Credit Risk Intelligence Platform** — Camada Silver

Este notebook transforma a tabela Bronze `credit_risk.bronze.bureau_balance` em uma tabela Silver tratada, padronizada e preparada para análise e Machine Learning.

## Pipeline

```
credit_risk.bronze.bureau_balance  →  credit_risk.silver.bureau_balance
```

## Sobre a tabela bureau_balance

A tabela `bureau_balance` contém o saldo mensal de cada crédito reportado pelo bureau. Cada registro representa o status de um crédito em um mês específico. A chave primária é a combinação de `SK_ID_BUREAU` + `MONTHS_BALANCE`.

- `SK_ID_BUREAU` — identificador do crédito no bureau (relaciona-se com `credit_risk.silver.bureau`)
- `MONTHS_BALANCE` — mês do saldo relativo à data da aplicação (0 = mês da aplicação, -1 = mês anterior, etc.)
- `STATUS` — status do crédito no mês (C=closed, X=unknown, 0=no DPD, 1-5=DPD crescente)

A tabela Silver permanece no **nível original dos registros** — nenhuma agregação é realizada.

## Transformações aplicadas

1. **Remoção de metadados Bronze** — colunas `_ingestion_timestamp` e `_source_file`
2. **Padronização de STATUS** — `trim()` para remover espaços extras
3. **Colunas de controle** — timestamp, versão, origem, hash
4. **Auditoria** — registro completo da transformação

## Regras

> A Bronze **NÃO é modificada**. Todas as transformações criam novas tabelas Silver.
> Nenhum registro é removido sem justificativa documentada.
> Nenhuma agregação é realizada — a tabela permanece no nível de registros originais.
> STATUS não é convertido para variável numérica (Feature Engineering é responsabilidade de outro notebook).

In [0]:
# ============================================================================
# CÉLULA 1 — Configuração, Imports e Parâmetros
# ============================================================================
from pyspark.sql import functions as F, types as T, Window
from datetime import datetime, timezone
import uuid

# ----------------------------------------------------------------------------
# Parâmetros do pipeline
# ----------------------------------------------------------------------------
PIPELINE_VERSION = "silver_v1.0"
NOTEBOOK_NAME = "07_silver_bureau_balance"
EXECUTION_ID = str(uuid.uuid4())
BATCH_ID = f"silver_bureau_bal_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}"
EXECUTION_TIMESTAMP = datetime.now(timezone.utc)

# ----------------------------------------------------------------------------
# Tabelas de origem (Bronze) e destino (Silver)
# ----------------------------------------------------------------------------
BRONZE_TABLE = "credit_risk.bronze.bureau_balance"
SILVER_TABLE = "credit_risk.silver.bureau_balance"
AUDIT_TABLE = "credit_risk.silver.audit_transformation"

# Tabela Silver de bureau (para integridade referencial)
SILVER_BUREAU = "credit_risk.silver.bureau"

# ----------------------------------------------------------------------------
# Colunas de metadados Bronze a remover na Silver
# ----------------------------------------------------------------------------
BRONZE_META_COLS = ["_ingestion_timestamp", "_source_file"]

# ----------------------------------------------------------------------------
# Criar schema Silver se não existir
# ----------------------------------------------------------------------------
spark.sql("CREATE SCHEMA IF NOT EXISTS credit_risk.silver")
print(f"Schema credit_risk.silver verificado/criado.")

# ----------------------------------------------------------------------------
# Dicionário para registrar transformações aplicadas (para auditoria)
# ----------------------------------------------------------------------------
TRANSFORMATION_LOG = []

def log_transform(table_name, step, description, records_affected=0):
    """Registra uma transformação aplicada para auditoria."""
    TRANSFORMATION_LOG.append({
        "table": table_name,
        "step": step,
        "description": description,
        "records_affected": records_affected,
    })

print(f"⏱️ Execution ID: {EXECUTION_ID}")
print(f"📦 Batch ID: {BATCH_ID}")
print(f"🔧 Pipeline Version: {PIPELINE_VERSION}")

In [0]:
# ============================================================================
# CÉLULA 2 — Leitura da Bronze e Inspeção do Schema
# ============================================================================
# Carrega o DataFrame Bronze (sem modificá-lo) e inspeciona o schema real.

df_bureau_bal_bronze = spark.table(BRONZE_TABLE)

# Métricas básicas
bronze_row_count = df_bureau_bal_bronze.count()
bronze_col_count = len(df_bureau_bal_bronze.columns)

print("=" * 70)
print("INSPEÇÃO INICIAL — BRONZE")
print("=" * 70)
print(f"\n📊 {BRONZE_TABLE}")
print(f"   Registros: {bronze_row_count:,}")
print(f"   Colunas: {bronze_col_count}")

# ----------------------------------------------------------------------------
# Schema detalhado (tipos e nullable)
# ----------------------------------------------------------------------------
sep = "─" * 70
print(f"\n{sep}")
print("SCHEMA — bureau_balance (tipos e nullable)")
print(sep)
for field in df_bureau_bal_bronze.schema.fields:
    print(f"   {field.name:<30} {field.dataType.simpleString():<12} nullable={field.nullable}")

# ----------------------------------------------------------------------------
# Verificar colunas-chave esperadas
# ----------------------------------------------------------------------------
print(f"\n{sep}")
print("COLUNAS-CHAVE")
print(sep)
key_cols = ["SK_ID_BUREAU", "MONTHS_BALANCE", "STATUS"]
for c in key_cols:
    if c in df_bureau_bal_bronze.columns:
        print(f"   ✅ {c}: presente")
    else:
        print(f"   ❌ {c}: AUSENTE")

print("\n✅ Leitura da Bronze concluída!")

In [0]:
# ============================================================================
# CÉLULA 3 — Data Quality Inicial (Bronze)
# ============================================================================
# Análise de completude (NULLs), valores distintos e estatísticas básicas.
# Usado como baseline para comparar Bronze → Silver.

sep = "─" * 70

# ----------------------------------------------------------------------------
# NULLs por coluna
# ----------------------------------------------------------------------------
null_exprs = [F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in df_bureau_bal_bronze.columns]
null_row = df_bureau_bal_bronze.agg(*null_exprs).collect()[0]
null_pairs = [(c, null_row[c]) for c in df_bureau_bal_bronze.columns if null_row[c] and null_row[c] > 0]

print(sep)
print(f"NULLs POR COLUNA — {BRONZE_TABLE}")
print(sep)
if null_pairs:
    for c, n in null_pairs:
        pct = n / bronze_row_count * 100
        print(f"   {c:<30} {n:>10,}  ({pct:.2f}%)")
else:
    print("   ✅ Nenhum NULL encontrado em qualquer coluna!")

# ----------------------------------------------------------------------------
# Valores distintos por coluna
# ----------------------------------------------------------------------------
print(f"\n{sep}")
print("VALORES DISTINCTOS POR COLUNA")
print(sep)
for c in df_bureau_bal_bronze.columns:
    if c not in BRONZE_META_COLS:
        d = df_bureau_bal_bronze.select(c).distinct().count()
        print(f"   {c:<30} {d:>10,}")

# ----------------------------------------------------------------------------
# Estatísticas numéricas (SK_ID_BUREAU e MONTHS_BALANCE)
# ----------------------------------------------------------------------------
numeric_cols = ["SK_ID_BUREAU", "MONTHS_BALANCE"]
print(f"\n{sep}")
print("ESTATÍSTICAS NUMÉRICAS (min, max, mean)")
print(sep)
for c in numeric_cols:
    stats = df_bureau_bal_bronze.select(c).summary("min", "max", "mean").collect()
    median = df_bureau_bal_bronze.approxQuantile(c, [0.5], 0.01)
    med_str = f"{median[0]:.1f}" if median else "N/A"
    mean_val = round(float(stats[2][c]), 2) if stats[2][c] else "N/A"
    print(f"   {c:<30} min={str(stats[0][c]):>15}  max={str(stats[1][c]):>15}  mean={str(mean_val):>15}  median={med_str}")

print("\n✅ Data Quality inicial concluída!")

In [0]:
# ============================================================================
# CÉLULA 4 — Análise de STATUS
# ============================================================================
# Inspeciona todos os valores distintos de STATUS com quantidade e percentual.
# Não substitui valores — apenas documenta a distribuição real.

sep = "─" * 70
print("=" * 70)
print("ANÁLISE DE STATUS")
print("=" * 70)

# ----------------------------------------------------------------------------
# Distribuição de STATUS
# ----------------------------------------------------------------------------
status_dist = df_bureau_bal_bronze.groupBy("STATUS").count().orderBy(F.desc("count")).collect()

print(f"\n{'STATUS':<10} {'Count':>15} {'Percentual':>12}  Significado")
print(sep)
status_meanings = {
    "C": "Closed (crédito encerrado)",
    "0": "No DPD (0 dias de atraso)",
    "X": "Unknown status (sem informação)",
    "1": "1-30 DPD (atraso 1-30 dias)",
    "2": "31-60 DPD (atraso 31-60 dias)",
    "3": "61-90 DPD (atraso 61-90 dias)",
    "4": "91-120 DPD (atraso 91-120 dias)",
    "5": "120+ DPD (atraso 120+ dias)",
}
for r in status_dist:
    s = str(r['STATUS'])
    pct = r['count'] / bronze_row_count * 100
    meaning = status_meanings.get(s, "Valor inesperado")
    print(f"{s:<10} {r['count']:>15,} {pct:>11.2f}%  {meaning}")

# ----------------------------------------------------------------------------
# Verificações
# ----------------------------------------------------------------------------
print(f"\n{sep}")
print("VERIFICAÇÕES")
print(sep)
status_null = df_bureau_bal_bronze.filter(F.col("STATUS").isNull()).count()
print(f"   NULL em STATUS: {status_null}")
print(f"   Valores distintos: {len(status_dist)}")

# Verificar se há valores inesperados (não previstos no dataset Home Credit)
expected_statuses = {"C", "0", "X", "1", "2", "3", "4", "5"}
actual_statuses = {str(r['STATUS']) for r in status_dist}
unexpected = actual_statuses - expected_statuses
if unexpected:
    print(f"   ⚠️ Valores inesperados: {unexpected}")
else:
    print(f"   ✅ Todos os {len(actual_statuses)} valores são esperados no dataset Home Credit")

print("\n✅ Análise de STATUS concluída!")

In [0]:
# ============================================================================
# CÉLULA 5 — Análise de MONTHS_BALANCE
# ============================================================================
# Analisa mínimo, máximo, distribuição e valores potencialmente inválidos.
# Preserva o significado original: 0 = mês da aplicação, negativos = meses anteriores.

sep = "─" * 70
print("=" * 70)
print("ANÁLISE DE MONTHS_BALANCE")
print("=" * 70)

# ----------------------------------------------------------------------------
# Estatísticas
# ----------------------------------------------------------------------------
mb_min = df_bureau_bal_bronze.select(F.min("MONTHS_BALANCE")).collect()[0][0]
mb_max = df_bureau_bal_bronze.select(F.max("MONTHS_BALANCE")).collect()[0][0]
mb_mean = df_bureau_bal_bronze.select(F.avg("MONTHS_BALANCE")).collect()[0][0]
mb_distinct = df_bureau_bal_bronze.select("MONTHS_BALANCE").distinct().count()
mb_null = df_bureau_bal_bronze.filter(F.col("MONTHS_BALANCE").isNull()).count()

print(f"\n   Min: {mb_min}")
print(f"   Max: {mb_max}")
print(f"   Média: {mb_mean:.2f}")
print(f"   Valores distintos: {mb_distinct}")
print(f"   NULL: {mb_null}")

# ----------------------------------------------------------------------------
# Valores potencialmente inválidos
# ----------------------------------------------------------------------------
print(f"\n{sep}")
print("VERIFICAÇÕES DE VALIDADE")
print(sep)

# MONTHS_BALANCE deveria ser <= 0 (0 = mês da aplicação, negativos = anteriores)
pos_count = df_bureau_bal_bronze.filter(F.col("MONTHS_BALANCE") > 0).count()
print(f"   Valores positivos (> 0): {pos_count} (esperado: 0 — todos devem ser <= 0)")
if pos_count == 0:
    print(f"   ✅ Todos os valores são <= 0 — consistentes com a semântica do dataset")
else:
    print(f"   ⚠️ {pos_count} valores positivos encontrados — investigar")

# ----------------------------------------------------------------------------
# Distribuição por faixas
# ----------------------------------------------------------------------------
print(f"\n{sep}")
print("DISTRIBUIÇÃO POR FAIXAS")
print(sep)
bins = [(0, 0), (-1, -6), (-7, -12), (-13, -24), (-25, -48), (-49, -96)]
for lo, hi in bins:
    cnt = df_bureau_bal_bronze.filter((F.col("MONTHS_BALANCE") >= lo) & (F.col("MONTHS_BALANCE") <= hi)).count()
    label = f"{lo}" if lo == hi else f"{hi} a {lo}"
    pct = cnt / bronze_row_count * 100
    print(f"   {label:<15} {cnt:>12,}  ({pct:.2f}%)")

print("\n✅ Análise de MONTHS_BALANCE concluída!")

In [0]:
# ============================================================================
# CÉLULA 6 — Análise de Duplicidades
# ============================================================================
# Analisa duplicidades completas e por chave composta (SK_ID_BUREAU + MONTHS_BALANCE).

sep = "─" * 70
print("=" * 70)
print("ANÁLISE DE DUPLICIDADES")
print("=" * 70)

# ----------------------------------------------------------------------------
# Duplicidade completa (linhas totalmente iguais)
# ----------------------------------------------------------------------------
full_dups = bronze_row_count - df_bureau_bal_bronze.dropDuplicates().count()
print(f"\n   Duplicidade completa: {full_dups} linhas totalmente duplicadas")

# ----------------------------------------------------------------------------
# Duplicidade por chave composta (SK_ID_BUREAU + MONTHS_BALANCE)
# ----------------------------------------------------------------------------
composite_dups = bronze_row_count - df_bureau_bal_bronze.select("SK_ID_BUREAU", "MONTHS_BALANCE").distinct().count()
print(f"   Duplicidade por (SK_ID_BUREAU + MONTHS_BALANCE): {composite_dups}")

if composite_dups == 0:
    print("   → A chave composta SK_ID_BUREAU + MONTHS_BALANCE é única — nenhum tratamento necessário")
else:
    print("   → ATENÇÃO: Chave composta tem duplicatas — investigar antes de tratar")

# ----------------------------------------------------------------------------
# Distribuição de registros por SK_ID_BUREAU
# ----------------------------------------------------------------------------
print(f"\n{sep}")
print("DISTRIBUIÇÃO DE REGISTROS POR SK_ID_BUREAU")
print(sep)

sk_distinct = df_bureau_bal_bronze.select("SK_ID_BUREAU").distinct().count()
per_sk_stats = df_bureau_bal_bronze.groupBy("SK_ID_BUREAU").count().select("count")
summary = per_sk_stats.summary("min", "max", "mean", "50%").collect()
for s in summary:
    print(f"   {s['summary']:<10}: {s['count']}")
print(f"\n   SK_ID_BUREAU distintos: {sk_distinct:,}")
print(f"   Média de registros por SK_ID_BUREAU: {bronze_row_count / sk_distinct:.1f}")

print("\n✅ Análise de duplicidades concluída!")

In [0]:
# ============================================================================
# CÉLULA 7 — Integridade Referencial
# ============================================================================
# Valida a relação SK_ID_BUREAU da bureau_balance contra a tabela Silver de bureau.
# Não exclui registros sem correspondência — apenas diagnostica a qualidade.

sep = "─" * 70
print("=" * 70)
print("INTEGRIDADE REFERENCIAL — bureau_balance vs bureau")
print("=" * 70)

bureau_bal_sk = df_bureau_bal_bronze.select("SK_ID_BUREAU").distinct()
bureau_bal_sk_count = bureau_bal_sk.count()

# ----------------------------------------------------------------------------
# Verificar se a tabela Silver de bureau existe
# ----------------------------------------------------------------------------
try:
    df_silver_bureau = spark.table(SILVER_BUREAU)
    bureau_silver_sk = df_silver_bureau.select("SK_ID_BUREAU").distinct()
    bureau_silver_count = bureau_silver_sk.count()
    print(f"\n   ✅ {SILVER_BUREAU}: {bureau_silver_count:,} SK_ID_BUREAU distintos")

    # ----------------------------------------------------------------------------
    # Integridade referencial
    # ----------------------------------------------------------------------------
    matched = bureau_bal_sk.join(bureau_silver_sk, "SK_ID_BUREAU", "inner").count()
    unmatched = bureau_bal_sk_count - matched
    print(f"\n📊 Bureau_balance SK_ID_BUREAU vs silver.bureau:")
    print(f"   Total bureau_balance: {bureau_bal_sk_count:,} SK_ID_BUREAU distintos")
    print(f"   Correspondidos: {matched:,} ({matched/bureau_bal_sk_count*100:.2f}%)")
    print(f"   Sem correspondência: {unmatched:,} ({unmatched/bureau_bal_sk_count*100:.2f}%)")
    if unmatched == 0:
        print(f"   → ✅ Todos os SK_ID_BUREAU da bureau_balance têm correspondência em silver.bureau")
    else:
        print(f"   → ⚠️ {unmatched} SK_ID_BUREAU sem correspondência (registros preservados)")

    # Verificar o sentido inverso: bureau SK_IDs without balance
    bureau_only = bureau_silver_sk.join(bureau_bal_sk, "SK_ID_BUREAU", "left_anti").count()
    print(f"\n📊 silver.bureau SK_ID_BUREAU sem bureau_balance:")
    print(f"   {bureau_only:,} de {bureau_silver_count:,} ({bureau_only/bureau_silver_count*100:.2f}%)")
    print(f"   → Esperado: alguns créditos no bureau não têm saldo mensal")

except Exception as e:
    print(f"   ⚠️ {SILVER_BUREAU}: tabela não encontrada — {e}")
    print(f"   Integridade referencial não verificada")

print("\n✅ Integridade referencial validada!")

In [0]:
# ============================================================================
# CÉLULA 8 — Funções de Transformação Reutilizáveis
# ============================================================================
# Funções modulares aplicadas na transformação Bronze → Silver.

def remove_bronze_metadata(df, table_name):
    """Remove colunas de metadados da Bronze (_ingestion_timestamp, _source_file).
    Serão substituídas por colunas de controle Silver."""
    cols_to_drop = [c for c in BRONZE_META_COLS if c in df.columns]
    if cols_to_drop:
        df = df.drop(*cols_to_drop)
        log_transform(table_name, "remove_metadata", f"Removidas colunas Bronze: {cols_to_drop}")
    return df


def standardize_categories(df, table_name):
    """Padroniza colunas categóricas: trim de espaços extras.
    Não altera semântica dos valores — apenas remove espaços à direita/esquerda."""
    string_cols = [f.name for f in df.schema.fields if f.dataType.simpleString() == "string"]
    for col_name in string_cols:
        df = df.withColumn(col_name, F.trim(F.col(col_name)))

    log_transform(table_name, "standardize_categories",
                  f"Trim aplicado em {len(string_cols)} colunas string")
    print(f"   ✅ Padronização: trim aplicado em {len(string_cols)} colunas string")
    return df


def add_control_columns(df, source_table):
    """Adiciona colunas de controle técnicas da Silver."""
    df = df.withColumn("silver_processing_timestamp", F.current_timestamp())
    df = df.withColumn("silver_processing_date", F.current_date())
    df = df.withColumn("silver_pipeline_version", F.lit(PIPELINE_VERSION))
    df = df.withColumn("source_table", F.lit(source_table))

    # record_hash: hash MD5 de todas as colunas de dados para rastreabilidade
    data_cols = [c for c in df.columns if c not in [
        "silver_processing_timestamp", "silver_processing_date",
        "silver_pipeline_version", "source_table"
    ]]
    hash_expr = F.concat_ws("||", *[F.coalesce(F.col(c).cast("string"), F.lit("NULL")) for c in data_cols])
    df = df.withColumn("record_hash", F.md5(hash_expr))

    print(f"   ✅ Colunas de controle adicionadas (timestamp, date, version, source, hash)")
    return df


def apply_silver_transformations(df, table_name, source_table, row_count):
    """Aplica todas as transformações Silver em sequência."""
    print(f"\n{'─' * 60}")
    print(f"🔧 Transformando: {table_name}")
    print(f"{'─' * 60}")

    # 1. Remover metadados Bronze
    df = remove_bronze_metadata(df, table_name)

    # 2. Padronizar categorias (trim)
    df = standardize_categories(df, table_name)

    # 3. Adicionar colunas de controle
    df = add_control_columns(df, source_table)

    print(f"   ✅ Transformações concluídas para {table_name}")
    return df


print("✅ Funções de transformação definidas!")

In [0]:
# ============================================================================
# CÉLULA 9 — Execução das Transformações
# ============================================================================
# Aplica as transformações Silver na tabela bureau_balance.
# O DataFrame Bronze original não é modificado.

EXEC_START = datetime.now(timezone.utc)

print("=" * 70)
print("TRANSFORMAÇÃO SILVER — bureau_balance")
print("=" * 70)
transform_start = datetime.now(timezone.utc)

df_bureau_bal_silver = apply_silver_transformations(
    df_bureau_bal_bronze, SILVER_TABLE, BRONZE_TABLE, bronze_row_count
)

transform_end = datetime.now(timezone.utc)
transform_duration = (transform_end - transform_start).total_seconds()
silver_row_count = df_bureau_bal_silver.count()
silver_col_count = len(df_bureau_bal_silver.columns)

print(f"\n   Bronze: {bronze_row_count:,} rows x {bronze_col_count} cols")
print(f"   Silver: {silver_row_count:,} rows x {silver_col_count} cols")
print(f"   Duração: {transform_duration:.1f}s")

print(f"\n{'=' * 70}")
print(f"⏱️ Tempo total de transformação: {transform_duration:.1f}s")
print(f"{'=' * 70}")

In [0]:
# ============================================================================
# CÉLULA 10 — Escrita da Tabela Silver (Delta Lake)
# ============================================================================
# Grava a tabela Silver usando mode("overwrite") com overwriteSchema.
# Isso é seguro porque a Silver é reconstruída a cada execução controlada.
# A Bronze NUNCA é sobrescrita.

print("=" * 70)
print("GRAVAÇÃO DA TABELA SILVER")
print("=" * 70)

print(f"\n📊 Gravando {SILVER_TABLE}...")
write_start = datetime.now(timezone.utc)

df_bureau_bal_silver.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable(SILVER_TABLE)

write_end = datetime.now(timezone.utc)
write_duration = (write_end - write_start).total_seconds()
print(f"   ✅ {SILVER_TABLE} gravada em {write_duration:.1f}s")
print(f"      Registros: {silver_row_count:,} | Colunas: {silver_col_count}")

EXEC_END = datetime.now(timezone.utc)
TOTAL_DURATION = (EXEC_END - EXEC_START).total_seconds()

print(f"\n{'=' * 70}")
print("✅ TABELA SILVER GRAVADA COM SUCESSO!")
print(f"{'=' * 70}")

In [0]:
# ============================================================================
# CÉLULA 11 — Auditoria da Transformação
# ============================================================================
# Cria/atualiza a tabela credit_risk.silver.audit_transformation
# Registra metadados da execução para rastreabilidade histórica (append mode).
from pyspark.sql.types import (StructType, StructField, StringType,
    IntegerType, DoubleType, TimestampType, LongType)

audit_record = {
    "execution_timestamp": EXECUTION_TIMESTAMP,
    "execution_id": EXECUTION_ID,
    "batch_id": BATCH_ID,
    "source_table": BRONZE_TABLE,
    "target_table": SILVER_TABLE,
    "source_row_count": bronze_row_count,
    "target_row_count": silver_row_count,
    "records_inserted": silver_row_count,
    "records_removed": 0,
    "records_changed": silver_row_count,
    "processing_duration_seconds": float(TOTAL_DURATION),
    "pipeline_version": PIPELINE_VERSION,
    "execution_status": "SUCCESS",
    "error_message": "",
}

audit_schema = StructType([
    StructField("execution_timestamp", TimestampType(), True),
    StructField("execution_id", StringType(), True),
    StructField("batch_id", StringType(), True),
    StructField("source_table", StringType(), True),
    StructField("target_table", StringType(), True),
    StructField("source_row_count", LongType(), True),
    StructField("target_row_count", LongType(), True),
    StructField("records_inserted", LongType(), True),
    StructField("records_removed", IntegerType(), True),
    StructField("records_changed", LongType(), True),
    StructField("processing_duration_seconds", DoubleType(), True),
    StructField("pipeline_version", StringType(), True),
    StructField("execution_status", StringType(), True),
    StructField("error_message", StringType(), True),
])

audit_df = spark.createDataFrame([audit_record], schema=audit_schema)

print(f"📊 Persistindo auditoria em {AUDIT_TABLE}...")
audit_df.write \
    .mode("append") \
    .format("delta") \
    .saveAsTable(AUDIT_TABLE)

print(f"✅ Auditoria registrada: 1 registro em {AUDIT_TABLE}")
print("\nRegistros de auditoria (últimos 10):")
display(spark.table(AUDIT_TABLE).orderBy(F.col("execution_timestamp").desc()).limit(10))

In [0]:
# ============================================================================
# CÉLULA 12 — Data Quality Pós-Transformação (Bronze vs Silver)
# ============================================================================
# Compara métricas de qualidade antes (Bronze) e depois (Silver).

def compute_dq_metrics(df, table_name):
    """Computa métricas de DQ: row_count, col_count, null_count, duplicate_count."""
    row_count = df.count()
    col_count = len(df.columns)

    # Total de NULLs (soma de todas as colunas)
    null_exprs = [F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)) for c in df.columns]
    total_nulls = df.agg(*null_exprs).collect()[0]
    null_sum = sum([total_nulls[i] for i in range(len(df.columns))])

    # Duplicatas por chave composta
    if "SK_ID_BUREAU" in df.columns and "MONTHS_BALANCE" in df.columns:
        dup_count = row_count - df.select("SK_ID_BUREAU", "MONTHS_BALANCE").distinct().count()
    else:
        dup_count = 0

    return {
        "table": table_name,
        "row_count": row_count,
        "col_count": col_count,
        "null_count": null_sum,
        "null_percentage": round(null_sum / (row_count * col_count) * 100, 2) if row_count > 0 else 0,
        "duplicate_count": dup_count,
    }

# ----------------------------------------------------------------------------
# Ler tabela Silver recém-criada
# ----------------------------------------------------------------------------
df_silver = spark.table(SILVER_TABLE)

# ----------------------------------------------------------------------------
# Métricas Bronze vs Silver
# ----------------------------------------------------------------------------
sep = "─" * 75
print("=" * 70)
print("DATA QUALITY: BRONZE vs SILVER")
print("=" * 70)

bronze_m = compute_dq_metrics(df_bureau_bal_bronze, BRONZE_TABLE)
silver_m = compute_dq_metrics(df_silver, SILVER_TABLE)

print(f"\n📊 bureau_balance")
print(f"{'Métrica':<30} {'Bronze':>15} {'Silver':>15} {'Delta':>15}")
print(sep)
for key in ["row_count", "col_count", "null_count", "null_percentage", "duplicate_count"]:
    b = bronze_m[key]
    s = silver_m[key]
    d = s - b
    print(f"{key:<30} {b:>15,} {s:>15,} {d:>+15,}")

# ----------------------------------------------------------------------------
# Verificações específicas
# ----------------------------------------------------------------------------
print(f"\n{sep}")
print("VERIFICAÇÕES ESPECÍFICAS")
print(sep)

# Colunas de controle presentes
control_cols = ["silver_processing_timestamp", "silver_processing_date",
                "silver_pipeline_version", "source_table", "record_hash"]
for c in control_cols:
    present = c in df_silver.columns
    print(f"   Coluna {c}: {'✅ presente' if present else '❌ ausente'}")

# Colunas Bronze removidas
print(f"\n   Colunas Bronze removidas:")
for c in BRONZE_META_COLS:
    present = c in df_silver.columns
    print(f"      {c}: {'❌ ainda presente' if present else '✅ removida'}")

# STATUS distribution preservada
print(f"\n   STATUS distribution (Silver):")
silver_status = df_silver.groupBy("STATUS").count().orderBy(F.desc("count")).collect()
for r in silver_status:
    print(f"      {r['STATUS']:<10} {r['count']:>12,}")

# MONTHS_BALANCE range preservado
silver_mb_min = df_silver.select(F.min("MONTHS_BALANCE")).collect()[0][0]
silver_mb_max = df_silver.select(F.max("MONTHS_BALANCE")).collect()[0][0]
print(f"\n   MONTHS_BALANCE (Silver): min={silver_mb_min}, max={silver_mb_max}")

# Chave composta uniqueness na Silver
silver_composite_dups = silver_row_count - df_silver.select("SK_ID_BUREAU", "MONTHS_BALANCE").distinct().count()
print(f"\n   (SK_ID_BUREAU + MONTHS_BALANCE) duplicatas na Silver: {silver_composite_dups} (esperado: 0)")

print("\n✅ Data Quality pós-transformação concluída!")

In [0]:
# ============================================================================
# CÉLULA 13 — Validação Final e Amostras
# ============================================================================
# Valida que a tabela Silver está correta e coerente com a Bronze.

sep = "─" * 70
print("=" * 70)
print("VALIDAÇÃO FINAL — TABELA SILVER")
print("=" * 70)

# ----------------------------------------------------------------------------
# Validação bureau_balance
# ----------------------------------------------------------------------------
print(f"\n📊 {SILVER_TABLE}")
print(sep)

# Comparar row count
assert silver_row_count == bronze_row_count, \
    f"Row count mismatch: Bronze={bronze_row_count} vs Silver={silver_row_count}"
print(f"   ✅ Row count: {silver_row_count:,} (igual à Bronze)")

# Comparar chave composta uniqueness
silver_dups = silver_row_count - df_silver.select("SK_ID_BUREAU", "MONTHS_BALANCE").distinct().count()
assert silver_dups == 0, f"Duplicatas encontradas: {silver_dups}"
print(f"   ✅ Chave composta (SK_ID_BUREAU + MONTHS_BALANCE): única (0 duplicatas)")

# Colunas Silver vs Bronze
print(f"   Colunas Bronze: {bronze_col_count}")
print(f"   Colunas Silver: {silver_col_count}")
print(f"   Colunas adicionadas: {silver_col_count - bronze_col_count}")
print(f"     - Removidas: 2 (metadados Bronze)")
print(f"     - Adicionadas: 5 colunas controle (timestamp, date, version, source, hash)")

# ----------------------------------------------------------------------------
# Amostra
# ----------------------------------------------------------------------------
print(f"\n{sep}")
print(f"AMOSTRA — {SILVER_TABLE} (primeiras 20 linhas)")
print(sep)

sample_cols = [
    "SK_ID_BUREAU", "MONTHS_BALANCE", "STATUS",
    "silver_processing_timestamp", "silver_pipeline_version",
    "source_table", "record_hash"
]
sample_cols = [c for c in sample_cols if c in df_silver.columns]
display(df_silver.select(*sample_cols).limit(20))

# ----------------------------------------------------------------------------
# Estatísticas de SK_ID_BUREAU na Silver
# ----------------------------------------------------------------------------
print(f"\n{sep}")
print("ESTATÍSTICAS — SK_ID_BUREAU (Silver)")
print(sep)

silver_sk_distinct = df_silver.select("SK_ID_BUREAU").distinct().count()
print(f"   SK_ID_BUREAU distintos: {silver_sk_distinct:,}")
print(f"   Registros por SK_ID_BUREAU (média): {silver_row_count / silver_sk_distinct:.1f}")

print("\n✅ Validação final concluída com sucesso!")

In [0]:
# ============================================================================
# CÉLULA 14 — Resumo Final da Execução
# ============================================================================
# Exibe um resumo completo da transformação.

print("=" * 60)
print("SILVER BUREAU_BALANCE - RESUMO")
print("=" * 60)

print(f"\nOrigem:\n  {BRONZE_TABLE}")
print(f"\nDestino:\n  {SILVER_TABLE}")
print(f"\nRegistros Bronze:\n  {bronze_row_count:,}")
print(f"\nRegistros Silver:\n  {silver_row_count:,}")
print(f"\nColunas:\n  Bronze: {bronze_col_count}")
print(f"  Silver: {silver_col_count}")
print(f"\nRegistros removidos:\n  0")
print(f"\nDuplicidades identificadas:\n  Completa: 0")
print(f"  Chave composta (SK_ID_BUREAU + MONTHS_BALANCE): 0")
print(f"\nDuplicidades removidas:\n  0 (não havia duplicidades)")
print(f"\nNULLs tratados:\n  0 (nenhum NULL encontrado na Bronze)")
print(f"\nValores inválidos identificados:\n  0 (nenhum valor inválido encontrado)")

# Integridade referencial
print(f"\nIntegridade referencial com silver.bureau:")
try:
    matched_final = bureau_bal_sk.join(bureau_silver_sk, "SK_ID_BUREAU", "inner").count()
    unmatched_final = bureau_bal_sk_count - matched_final
    print(f"  Correspondidos: {matched_final:,} ({matched_final/bureau_bal_sk_count*100:.2f}%)")
    print(f"  Sem correspondência: {unmatched_final:,} ({unmatched_final/bureau_bal_sk_count*100:.2f}%)")
except Exception:
    print(f"  Verificação não realizada")

# Transformações aplicadas
print(f"\nRegras aplicadas ({len(TRANSFORMATION_LOG)}):")
for t in TRANSFORMATION_LOG:
    print(f"  • {t['step']}: {t['description']}")

print(f"\nStatus:\n  SUCCESS")
print(f"\nTempo:\n  {TOTAL_DURATION:.1f} segundos")
print(f"\n⏱️ Execution ID: {EXECUTION_ID}")
print(f"📦 Batch ID: {BATCH_ID}")
print(f"🔧 Pipeline: {PIPELINE_VERSION}")
print(f"\n{'=' * 60}")
print("✅ PIPELINE SILVER BUREAU_BALANCE CONCLUÍDO COM SUCESSO!")
print(f"{'=' * 60}")

## Transformações Aplicadas — Documentação

### 1. Remoção de metadados Bronze
Colunas `_ingestion_timestamp` e `_source_file` removidas (substituídas por colunas de controle Silver).

### 2. Padronização de STATUS
- `trim()` aplicado na coluna `STATUS` para remover espaços extras
- Não houve alteração semântica dos valores
- STATUS não foi convertido para variável numérica (Feature Engineering é responsabilidade de outro notebook)

### 3. Tratamento de NULLs
- **Nenhum NULL encontrado** em qualquer coluna da Bronze
- Nenhum tratamento de NULL foi necessário

### 4. Duplicidades
- 0 linhas totalmente duplicadas
- 0 duplicatas na chave composta (SK_ID_BUREAU + MONTHS_BALANCE)
- A chave composta é naturalmente única — cada crédito tem no máximo um registro por mês
- Nenhuma deduplicação foi necessária

### 5. STATUS — Distribuição e Significado

| STATUS | Count | Percentual | Significado |
|--------|-------|------------|------------|
| C | 13.646.993 | 49,99% | Closed (crédito encerrado) |
| 0 | 7.499.507 | 27,47% | No DPD (0 dias de atraso) |
| X | 5.810.482 | 21,28% | Unknown status (sem informação) |
| 1 | 242.347 | 0,89% | 1-30 DPD (atraso 1-30 dias) |
| 5 | 62.406 | 0,23% | 120+ DPD (atraso 120+ dias) |
| 2 | 23.419 | 0,09% | 31-60 DPD (atraso 31-60 dias) |
| 3 | 8.924 | 0,03% | 61-90 DPD (atraso 61-90 dias) |
| 4 | 5.847 | 0,02% | 91-120 DPD (atraso 91-120 dias) |

- Todos os 8 valores são esperados no dataset Home Credit
- Nenhum valor inesperado ou NULL encontrado

### 6. MONTHS_BALANCE
- Min: -96 (96 meses antes da aplicação)
- Max: 0 (mês da aplicação)
- Média: -30,74
- 97 valores distintos (um para cada mês no intervalo)
- Nenhum valor positivo encontrado (todos <= 0, consistente com a semântica)
- Nenhuma conversão de representação foi aplicada

### 7. Integridade Referencial
- bureau_balance.SK_ID_BUREAU → silver.bureau.SK_ID_BUREAU
- 100% dos SK_ID_BUREAU da bureau_balance correspondem ao silver.bureau
- Nenhum registro foi removido por falta de correspondência

### 8. Colunas de controle Silver
| Coluna | Tipo | Descrição |
|--------|------|------------|
| `silver_processing_timestamp` | timestamp | Momento da transformação |
| `silver_processing_date` | date | Data da transformação |
| `silver_pipeline_version` | string | Versão do pipeline (`silver_v1.0`) |
| `source_table` | string | Tabela de origem Bronze |
| `record_hash` | string | Hash MD5 de todos os campos para rastreabilidade |

### 9. Identificadores
- `SK_ID_BUREAU`: 817.395 valores distintos, 0 NULL
- Chave composta `SK_ID_BUREAU + MONTHS_BALANCE`: única (0 duplicatas)
- Média de 33,4 registros por SK_ID_BUREAU (mediana: 26, máximo: 97)